# RQFormer-2 Kaggle Workflow

Notebook này chạy toàn bộ quy trình trên Kaggle: chuẩn bị source/data, setup môi trường, train hoặc load checkpoint, test, vẽ biểu đồ, visualize bbox, ghi summary và zip kết quả.


## Metric Và Ghi Chú

Notebook gọi trực tiếp `tools/train.py` và `tools/test.py`, không viết lại model. AP50/AP75 được tính tại ngưỡng IoU 0.50 và 0.75.

$$
IoU(\hat{b}, b) \ge 0.5
$$

Auto LR bắt đầu cao hơn LR gốc rồi giảm dần bằng cosine decay:

$$
LR_t = LR_{min} + \frac{1}{2}(LR_{peak}-LR_{min})\left(1 + \cos\frac{\pi t}{T}\right)
$$

`EVAL_IOU_THRS = [0.5, 0.75]` giúp log có AP50, AP75 và mAP. `SCORE_THR` chỉ dùng để lọc ảnh visualization.


## 1. Cấu Hình Chạy

Chỉnh các biến chính ở đây trước khi Run All: dataset, GPU, batch, LR, checkpoint, số epoch và các bước workflow.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import json

import pandas as pd

REPO_GIT_URL = "https://github.com/hungnp-dev/RQFormer.git"
REPO_DIR = Path("/kaggle/working/RQFormer")
REPO_INPUT_FALLBACK = Path("/kaggle/input/rqformer-source/RQFormer")

DATA_ROOT = Path("/kaggle/input/datasets/jks010/rqformer-dataset/data")
DATA_INPUTS = {
    "dior": DATA_ROOT / "DIOR",
    "dotav1_0": DATA_ROOT / "split_ss_dota",
    "dotav1_5": DATA_ROOT / "split_ss_dota1_5",
}

CKPT_INPUT = None
CKPT_DIR = Path("/kaggle/working/pth")
WORK_ROOT = Path("/kaggle/working/work_dirs")

DATASETS = "dior"

DEVICE = "auto"
NUM_GPUS = "auto"
BATCH_SIZE = "auto"
NUM_WORKERS = "auto"
MAX_EPOCHS = "auto"
LR = "auto"

BASE_LR = 2.5e-5
REFERENCE_GLOBAL_BATCH = 2
AUTO_LR_SCALE = 1.0
MIN_LR = 1e-6
MAX_LR = 3.5e-5
LR_SCHEDULE = "cosine_decay"
LR_WARMUP_ITERS = 500
GRAD_CLIP_MAX_NORM = 0.5
AUTO_LR_RETRY_ON_FAIL = 1
AUTO_LR_RETRY_FACTOR = 0.5
AUTO_LR_MAX_RETRIES = 4

SAMPLE_IMAGES = 20
BBOX_GROUP_BY_CLASS = 1
BBOX_IMAGES_PER_CLASS = 2
BBOX_CANDIDATES_PER_CLASS = 12
BBOX_GRID_COLS = 2
BBOX_IMAGES_PER_FIGURE = 4
SCORE_THR = 0.3
EVAL_IOU_THRS = [0.5, 0.75]

# Optional test-time postprocess. None keeps the original RQFormer behavior.
TEST_SCORE_THR = None
TEST_NMS_TYPE = None  # e.g. "nms_rotated"
TEST_NMS_IOU = 0.5
TEST_NMS_PRE = 1000
TEST_MAX_PER_IMG = None
FORCE = 1
MASTER_PORT = 29500

EARLY_STOP = 1
EARLY_STOP_MONITOR = "dota/mAP"
EARLY_STOP_PATIENCE = 2
EARLY_STOP_MIN_DELTA = 0.001

RUN_SETUP = 1
RUN_TRAIN = 1
RUN_TEST = 1
RUN_CHARTS = 1
RUN_CONFUSION = 1
RUN_BBOX = 1
RUN_SUMMARY = 1
AUTO_DISPLAY_OUTPUTS = 1

pd.DataFrame([
    ("repo", str(REPO_DIR)),
    ("data_root", str(DATA_ROOT)),
    ("datasets", DATASETS),
    ("device", DEVICE),
    ("num_gpus", NUM_GPUS),
    ("batch_per_gpu", BATCH_SIZE),
    ("workers_per_gpu", NUM_WORKERS),
    ("max_epochs", MAX_EPOCHS),
    ("learning_rate", LR),
    ("lr_schedule", LR_SCHEDULE),
    ("lr_warmup_iters", LR_WARMUP_ITERS),
    ("auto_lr_scale", AUTO_LR_SCALE),
    ("max_lr", MAX_LR),
    ("min_lr", MIN_LR),
    ("grad_clip_max_norm", GRAD_CLIP_MAX_NORM),
    ("auto_lr_retry_on_fail", AUTO_LR_RETRY_ON_FAIL),
    ("auto_lr_max_retries", AUTO_LR_MAX_RETRIES),
    ("score_thr_visualization", SCORE_THR),
    ("eval_iou_thrs", EVAL_IOU_THRS),
    ("test_score_thr", TEST_SCORE_THR),
    ("test_nms_type", TEST_NMS_TYPE),
    ("test_nms_iou", TEST_NMS_IOU),
    ("test_max_per_img", TEST_MAX_PER_IMG),
    ("sample_images", SAMPLE_IMAGES),
    ("bbox_group_by_class", BBOX_GROUP_BY_CLASS),
    ("bbox_images_per_class", BBOX_IMAGES_PER_CLASS),
    ("bbox_candidates_per_class", BBOX_CANDIDATES_PER_CLASS),
    ("bbox_grid", f"{BBOX_GRID_COLS} x {BBOX_IMAGES_PER_FIGURE // BBOX_GRID_COLS}"),
    ("force", FORCE),
    ("auto_display_outputs", AUTO_DISPLAY_OUTPUTS),
], columns=["setting", "value"])


## 2. Danh Sách Dataset

Khai báo từng job gồm tên dataset, config, checkpoint, đường dẫn data và thư mục ảnh dùng để visualize.


In [ ]:
JOBS = {
    "dior": {
        "title": "RQFormer | DIOR-R | R50 | 3x | Query 500 | t0.85",
        "dataset_label": "DIOR-R",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.pth",
        "data_link": Path("data/DIOR"),
        "data_input": DATA_INPUTS["dior"],
        "image_source": Path("data/DIOR/JPEGImages-test"),
    },
    "dotav1_0": {
        "title": "RQFormer | DOTA-v1.0 | R50 | 2x | Query 500 | t0.9",
        "dataset_label": "DOTA-v1.0",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.0.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.0.pth",
        "data_link": Path("data/split_ss_dota"),
        "data_input": DATA_INPUTS["dotav1_0"],
        "image_source": Path("data/split_ss_dota/trainval/images"),
    },
    "dotav1_5": {
        "title": "RQFormer | DOTA-v1.5 | R50 | 2x | Query 500 | t0.9",
        "dataset_label": "DOTA-v1.5",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.5.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.5.pth",
        "data_link": Path("data/split_ss_dota1_5"),
        "data_input": DATA_INPUTS["dotav1_5"],
        "image_source": Path("data/split_ss_dota1_5/trainval/images"),
    },
}

def selected_names():
    if DATASETS == "all":
        return list(JOBS)
    names = [x.strip() for x in DATASETS.split(",") if x.strip()]
    unknown = [x for x in names if x not in JOBS]
    if unknown:
        raise ValueError(f"Unknown DATASETS: {unknown}. Valid: {list(JOBS)}")
    return names

print("Selected jobs:", selected_names())


## 3. Chuẩn Bị Source Và Data

Clone hoặc copy source, link dataset vào đúng cấu trúc repo, patch base config local và sửa script bbox nếu cần.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path



def copy_repo_fallback():
    if not REPO_INPUT_FALLBACK.exists():
        raise RuntimeError(
            "Không clone được GitHub và cũng không thấy REPO_INPUT_FALLBACK.\n"
            "Hãy bật Internet cho Kaggle hoặc upload source repo "
            f"tại {REPO_INPUT_FALLBACK}"
        )
    ignore = shutil.ignore_patterns(".git", "work_dirs", "__pycache__", "*.pyc", ".ipynb_checkpoints")
    shutil.copytree(REPO_INPUT_FALLBACK, REPO_DIR, ignore=ignore)
    print(f"[OK] Copied repo fallback: {REPO_INPUT_FALLBACK} -> {REPO_DIR}")


def clone_repo():
    if REPO_DIR.exists():
        print(f"[OK] Repo exists: {REPO_DIR}")
        return
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

    public_cmd = ["git", "clone", REPO_GIT_URL, str(REPO_DIR)]
    print("$", " ".join(public_cmd))
    try:
        subprocess.check_call(public_cmd)
        print(f"[OK] Cloned public repo")
        return
    except subprocess.CalledProcessError as exc:
        print(f"[WARN] Public clone failed: {exc.returncode}")

    copy_repo_fallback()


def link_or_copy(src: Path, dst: Path):
    if dst.exists() or dst.is_symlink():
        print(f"[OK] Exists: {dst}")
        return
    if not src.exists():
        raise FileNotFoundError(f"Không thấy input: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        dst.symlink_to(src, target_is_directory=src.is_dir())
        print(f"[OK] Symlink: {dst} -> {src}")
    except OSError:
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        print(f"[OK] Copied: {src} -> {dst}")


def strip_bom_from_text_files():
    """Remove UTF-8 BOM that breaks Python 3.7 ast.parse on Kaggle."""
    patterns = ["configs/**/*.py", "projects/**/*.py", "tools/**/*.py", "tools/**/*.sh"]
    fixed = []
    for pattern in patterns:
        for path in REPO_DIR.glob(pattern):
            if not path.is_file():
                continue
            raw = path.read_bytes()
            if raw.startswith(b"\xef\xbb\xbf"):
                path.write_bytes(raw[3:])
                fixed.append(path)
    if fixed:
        print("[OK] Removed UTF-8 BOM from:")
        for path in fixed:
            print("  ", path.relative_to(REPO_DIR))
    else:
        print("[OK] No UTF-8 BOM found in config/source files")


def patch_configs_to_local_base():
    replacements = {
        "mmrotate::_base_/datasets/dior.py": "../../../configs/_base_/datasets/dior.py",
        "mmrotate::_base_/datasets/dota.py": "../../../configs/_base_/datasets/dota.py",
        "mmrotate::_base_/datasets/dotav15.py": "../../../configs/_base_/datasets/dotav15.py",
        "mmrotate::_base_/schedules/schedule_3x.py": "../../../configs/_base_/schedules/schedule_3x.py",
        "mmrotate::_base_/default_runtime.py": "../../../configs/_base_/default_runtime.py",
    }
    for name in JOBS:
        cfg = REPO_DIR / JOBS[name]["config"]
        text = cfg.read_text(encoding="utf-8")
        new_text = text
        for old, new in replacements.items():
            new_text = new_text.replace(old, new)
        if new_text != text:
            cfg.write_text(new_text, encoding="utf-8")
            print(f"[OK] Patched base paths: {cfg}")



def patch_predict_bbox_postprocess():
    path = REPO_DIR / "projects/RQFormer/rroiformer/rroiformer_decoder.py"
    if not path.exists():
        print("[SKIP PATCH TEST POSTPROCESS] Missing", path)
        return
    text = path.read_text(encoding="utf-8")
    if "score_thr = float(rcnn_test_cfg.get('score_thr', 0.0))" in text:
        print("[OK] Test postprocess already supports score_thr/nms/max_per_img")
        return

    old = """        topk_inds_list = []
        results_list = []
        for img_id in range(len(batch_img_metas)):
            cls_score_per_img = cls_score_[img_id]
            scores_per_img, topk_inds = cls_score_per_img.flatten(0, 1).topk(
                batch_start_index[img_id+1]-batch_start_index[img_id], sorted=False)
            labels_per_img = topk_inds % num_classes
            bboxes_per_img = proposal_list[img_id][topk_inds // num_classes]
            topk_inds_list.append(topk_inds)
            if rescale and bboxes_per_img.size(0) > 0:
"""
    new = """        topk_inds_list = []
        results_list = []
        score_thr = float(rcnn_test_cfg.get('score_thr', 0.0)) if rcnn_test_cfg else 0.0
        nms_cfg = rcnn_test_cfg.get('nms', None) if rcnn_test_cfg else None
        nms_pre = int(rcnn_test_cfg.get('nms_pre', -1)) if rcnn_test_cfg else -1
        max_per_img = int(rcnn_test_cfg.get('max_per_img', initial_num_query)) if rcnn_test_cfg else initial_num_query
        for img_id in range(len(batch_img_metas)):
            cls_score_per_img = cls_score_[img_id]
            flat_scores = cls_score_per_img.flatten(0, 1)
            pre_topk = flat_scores.numel()
            if nms_cfg is None:
                pre_topk = min(max_per_img, pre_topk)
            elif nms_pre > 0:
                pre_topk = min(nms_pre, pre_topk)
            scores_per_img, topk_inds = flat_scores.topk(pre_topk, sorted=True)
            labels_per_img = topk_inds % num_classes
            bboxes_per_img = proposal_list[img_id][topk_inds // num_classes]

            if score_thr > 0:
                valid = scores_per_img >= score_thr
                scores_per_img = scores_per_img[valid]
                labels_per_img = labels_per_img[valid]
                bboxes_per_img = bboxes_per_img[valid]
                topk_inds = topk_inds[valid]

            if nms_cfg is not None and bboxes_per_img.size(0) > 0:
                _, keep_idxs = batched_nms(
                    bboxes_per_img, scores_per_img, labels_per_img, nms_cfg)
                if max_per_img > 0:
                    keep_idxs = keep_idxs[:max_per_img]
                scores_per_img = scores_per_img[keep_idxs]
                labels_per_img = labels_per_img[keep_idxs]
                bboxes_per_img = bboxes_per_img[keep_idxs]
                topk_inds = topk_inds[keep_idxs]

            topk_inds_list.append(topk_inds)
            if rescale and bboxes_per_img.size(0) > 0:
"""
    if old not in text:
        print("[WARN PATCH TEST POSTPROCESS] Expected predict_bbox block not found")
        return
    path.write_text(text.replace(old, new), encoding="utf-8")
    print("[OK] Patched predict_bbox test postprocess")


def prepare_data_and_ckpts():
    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"Không thấy DATA_ROOT: {DATA_ROOT}")
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    for name in selected_names():
        job = JOBS[name]
        link_or_copy(job["data_input"], REPO_DIR / job["data_link"])
        if CKPT_INPUT is None:
            print("[SKIP CKPT INPUT] Không dùng checkpoint ngoài; train từ đầu hoặc resume từ work_dirs.")
            continue
        ckpt_src = CKPT_INPUT / job["ckpt"]
        ckpt_dst = CKPT_DIR / job["ckpt"]
        if ckpt_src.exists():
            link_or_copy(ckpt_src, ckpt_dst)
        else:
            print(f"[WARN] Không thấy checkpoint gốc: {ckpt_src}; train từ đầu vẫn chạy được.")

clone_repo()
os.chdir(REPO_DIR)
strip_bom_from_text_files()
patch_configs_to_local_base()
patch_predict_bbox_postprocess()
prepare_data_and_ckpts()
print("cwd =", Path.cwd())


## 4. Setup Môi Trường

Kiểm tra và cài các dependency OpenMMLab/MMRotate còn thiếu, sau đó hiển thị phiên bản package.


In [ ]:
import importlib.util
import sys


def has_module(name):
    return importlib.util.find_spec(name) is not None


def run(cmd, cwd=REPO_DIR, env=None):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=cwd, env=env)


def env_ready():
    try:
        import torch, mmcv, mmdet, mmengine, mmrotate
        print("Environment ready:", torch.__version__, mmcv.__version__, mmdet.__version__, mmrotate.__version__)
        return True
    except Exception as exc:
        print("Environment not ready:", repr(exc))
        return False

if RUN_SETUP:
    if not env_ready():
        run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
        req_build = REPO_DIR / "requirements/build.txt"
        if req_build.exists():
            run([sys.executable, "-m", "pip", "install", "-r", str(req_build)])
        run([sys.executable, "-m", "pip", "install", "packaging", "matplotlib", "pycocotools", "six", "terminaltables", "scipy", "scikit-learn", "imagecorruptions"])
        if not has_module("openmim"):
            run([sys.executable, "-m", "pip", "install", "-U", "openmim"])
        if not has_module("mmcv"):
            run([sys.executable, "-m", "mim", "install", "mmcv>=2.0.0rc2,<2.1.0"])
        if not has_module("mmengine"):
            run([sys.executable, "-m", "pip", "install", "mmengine>=0.1.0"])
        if not has_module("mmdet"):
            run([sys.executable, "-m", "pip", "install", "mmdet>=3.0.0rc5,<3.1.0"])
        run([sys.executable, "-m", "pip", "install", "-v", "-e", "."])
    env_ready()
else:
    print("[SKIP SETUP]")


## 5. Mapping Cải Tiến

Notebook dùng trực tiếp code gốc và các module cải tiến trong source, không viết lại kiến trúc model trong notebook.

| Khối | Phương pháp gốc | Phương pháp cải tiến | File chính | Vai trò |
|---|---|---|---|---|
| RRoI attention | Attention dựa trên query và đặc trưng RRoI. | Geometry-aware RRoI attention. | `projects/RQFormer/rroiformer/rroiattention.py` | Gốc trích đặc trưng vùng xoay; cải tiến thêm hình học hộp xoay vào query/attention. |
| Decoder query | Query tương tác qua decoder layer chuẩn. | Pairwise query geometry relation. | `projects/RQFormer/rroiformer/rroiformer_decoder_layer.py` | Gốc cập nhật query theo feature; cải tiến cho query trao đổi quan hệ vị trí, kích thước và góc. |
| Dataset/config | Config RQFormer cho DIOR-R/DOTA. | Bật đúng biến thể RQFormer-2 theo từng dataset. | `projects/RQFormer/configs/*.py` | Gốc định nghĩa train/test; cải tiến đảm bảo các module geometry được bật và mapping đúng dataset. |
| Train/test workflow | Chạy `tools/train.py` và `tools/test.py`. | Resume, auto LR, early stop, AP50/AP75, visualization theo class. | `notebooks/kaggle_rqformer2_finetune.ipynb` | Gốc giữ pipeline OpenMMLab; cải tiến tự động hóa chạy Kaggle và trình bày kết quả báo cáo. |

**Geometry injection trong RRoI attention**

$$
q'_i = q_i + g(b_i)
$$

| Ký hiệu | Ý nghĩa |
|---|---|
| $q_i$ | Query gốc của object thứ $i$. |
| $b_i$ | Hộp xoay của object thứ $i$, gồm vị trí, kích thước và góc. |
| $g(b_i)$ | Embedding hình học học được từ hộp xoay. |
| $q'_i$ | Query sau khi thêm thông tin hình học. |

**Quan hệ hình học giữa các query**

$$
q'_i = q_i + \alpha \sum_{j} \operatorname{softmax}\left(\phi(r_{ij})\right) q_j
$$

| Ký hiệu | Ý nghĩa |
|---|---|
| $q_j$ | Query của object thứ $j$ dùng làm ngữ cảnh cho query $i$. |
| $r_{ij}$ | Quan hệ hình học tương đối giữa hai hộp $i$ và $j$. |
| $\phi(r_{ij})$ | Hàm biến đổi quan hệ hình học thành trọng số attention. |
| $\alpha$ | Hệ số học được để kiểm soát mức ảnh hưởng của quan hệ hình học. |
| $\operatorname{softmax}$ | Chuẩn hóa trọng số quan hệ giữa các query. |


## 6. Hàm Runtime

Các hàm này tự nhận GPU/batch/workers/LR, tạo lệnh DDP, quản lý checkpoint, chạy train/test và các bước phân tích. LR cao sẽ tự giảm theo scheduler; nếu train fail, notebook thử giảm LR rồi chạy lại.


In [ ]:

def is_auto(value):
    return isinstance(value, str) and value.lower() == "auto"


def detected_gpu_count():
    try:
        import torch
        count = torch.cuda.device_count()
        if count:
            return count
    except Exception:
        pass
    try:
        result = subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True, check=False)
        lines = [line for line in result.stdout.splitlines() if line.strip().startswith("GPU ")]
        return len(lines)
    except Exception:
        return 0


def resolved_device():
    if not is_auto(DEVICE):
        return str(DEVICE)
    count = detected_gpu_count()
    if count <= 0:
        return ""
    return ",".join(str(i) for i in range(count))


def resolved_num_gpus():
    if not is_auto(NUM_GPUS):
        return int(NUM_GPUS)
    dev = resolved_device()
    if not dev:
        return 0
    return len([x for x in dev.split(",") if x.strip()])


def resolved_batch_size():
    if not is_auto(BATCH_SIZE):
        return int(BATCH_SIZE)
    gpu_count = resolved_num_gpus()
    if gpu_count <= 0:
        return 1
    # RQFormer is numerically sensitive on DIOR-R; batch/GPU=2 is the stable fast default on T4.
    return 2


def resolved_num_workers():
    if not is_auto(NUM_WORKERS):
        return int(NUM_WORKERS)
    cpu_count = os.cpu_count() or 2
    gpu_count = max(1, resolved_num_gpus())
    # This is per DDP process. Keep total workers near CPU count to avoid Kaggle CPU contention.
    return max(1, min(2, cpu_count // gpu_count))


def resolved_max_epochs():
    if not is_auto(MAX_EPOCHS):
        return MAX_EPOCHS
    # Good balance for Kaggle 30h: enough for strong DIOR-R results, leaves time for test/visualization/zip.
    return 24


def global_batch_size():
    return resolved_batch_size() * max(1, resolved_num_gpus())


def resolved_lr():
    if isinstance(LR, str) and LR.lower() == "auto":
        lr = BASE_LR * (global_batch_size() / REFERENCE_GLOBAL_BATCH) * AUTO_LR_SCALE
        return min(max(lr, MIN_LR), MAX_LR)
    return LR


def py_value(value):
    return repr(value)


def runtime_config_path(job, root):
    runtime_dir = root / "runtime_configs"
    runtime_dir.mkdir(parents=True, exist_ok=True)
    return runtime_dir / (Path(job["config"]).stem + "_runtime.py")


def write_runtime_config(job, root, load_from=None):
    max_epochs = int(resolved_max_epochs())
    lines = [
        f"_base_ = {py_value(str(REPO_DIR / job['config']))}",
        "",
        f"train_cfg = dict(max_epochs={max_epochs})",
        "optim_wrapper = dict(",
        f"    optimizer=dict(lr={float(resolved_lr())}),",
        f"    clip_grad=dict(max_norm={float(GRAD_CLIP_MAX_NORM)}, norm_type=2),",
        ")",
        "",
    ]

    if LR_SCHEDULE and str(LR_SCHEDULE).lower() not in {"none", "off", "false"}:
        scheduler_lines = ["param_scheduler = ["]
        if int(LR_WARMUP_ITERS) > 0:
            scheduler_lines += [
                "    dict(",
                "        type='LinearLR',",
                "        start_factor=0.1,",
                "        by_epoch=False,",
                "        begin=0,",
                f"        end={int(LR_WARMUP_ITERS)},",
                "    ),",
            ]
        scheduler_lines += [
            "    dict(",
            "        type='CosineAnnealingLR',",
            f"        eta_min={float(MIN_LR)},",
            "        begin=0,",
            f"        end={max_epochs},",
            f"        T_max={max_epochs},",
            "        by_epoch=True,",
            "        convert_to_iter_based=True,",
            "    ),",
            "]",
            "",
        ]
        lines += scheduler_lines

    if EVAL_IOU_THRS:
        lines += [
            f"val_evaluator = dict(iou_thrs={py_value([float(x) for x in EVAL_IOU_THRS])})",
            f"test_evaluator = dict(iou_thrs={py_value([float(x) for x in EVAL_IOU_THRS])})",
            "",
        ]

    test_rcnn_cfg = {}
    if TEST_MAX_PER_IMG is not None:
        test_rcnn_cfg["max_per_img"] = int(TEST_MAX_PER_IMG)
    if TEST_SCORE_THR is not None:
        test_rcnn_cfg["score_thr"] = float(TEST_SCORE_THR)
    if TEST_NMS_TYPE:
        test_rcnn_cfg["nms_pre"] = int(TEST_NMS_PRE)
        test_rcnn_cfg["nms"] = dict(type=str(TEST_NMS_TYPE), iou_threshold=float(TEST_NMS_IOU))
    if test_rcnn_cfg:
        lines += [
            f"model = dict(test_cfg=dict(rcnn={py_value(test_rcnn_cfg)}))",
            "",
        ]

    if load_from:
        lines += [f"load_from = {py_value(str(load_from))}", ""]

    if EARLY_STOP:
        lines += [
            "custom_hooks = [dict(",
            "    type='EarlyStoppingHook',",
            f"    monitor={py_value(EARLY_STOP_MONITOR)},",
            "    rule='greater',",
            f"    patience={int(EARLY_STOP_PATIENCE)},",
            f"    min_delta={float(EARLY_STOP_MIN_DELTA)},",
            "    strict=False,",
            ")]",
            "",
        ]

    path = runtime_config_path(job, root)
    path.write_text("\n".join(lines), encoding="utf-8")
    return path



def runtime_table():
    return pd.DataFrame([{
        "device": resolved_device() or "cpu",
        "num_gpus": resolved_num_gpus(),
        "batch_per_gpu": resolved_batch_size(),
        "global_batch": global_batch_size(),
        "workers_per_gpu": resolved_num_workers(),
        "max_epochs": resolved_max_epochs(),
        "peak_lr": resolved_lr(),
        "lr_schedule": LR_SCHEDULE,
        "min_lr": MIN_LR,
        "warmup_iters": LR_WARMUP_ITERS,
        "grad_clip": GRAD_CLIP_MAX_NORM,
        "early_stop": bool(EARLY_STOP),
        "eval_iou_thrs": EVAL_IOU_THRS,
    }])

def use_distributed():
    return resolved_num_gpus() > 1


def ddp_cmd(script, *args):
    if use_distributed():
        return [
            sys.executable, "-m", "torch.distributed.run",
            "--nproc_per_node", str(resolved_num_gpus()),
            "--master_port", str(MASTER_PORT),
            script, *map(str, args),
            "--launcher", "pytorch",
        ]
    return [sys.executable, script, *map(str, args)]


def per_gpu_note():
    global_batch = global_batch_size()
    print(f"[GPU CONFIG] CUDA_VISIBLE_DEVICES={resolved_device()} | NUM_GPUS={resolved_num_gpus()} | batch/GPU={resolved_batch_size()} | global batch={global_batch}")
    print(f"[WORKERS] NUM_WORKERS={resolved_num_workers()} per process")
    print(f"[EPOCHS] MAX_EPOCHS={resolved_max_epochs()}")
    print(f"[LR CONFIG] LR={LR} | peak_lr={resolved_lr()} | schedule={LR_SCHEDULE} | eta_min={MIN_LR}")
    print(f"[EARLY STOP] enabled={EARLY_STOP} | monitor={EARLY_STOP_MONITOR} | patience={EARLY_STOP_PATIENCE}")


def ckpt_name_for(job):
    return job["ckpt"]


def latest_ckpt(path: Path, ckpt_name=None):
    path = Path(path)
    if not path.exists():
        return None
    if ckpt_name:
        named = path / ckpt_name
        if named.exists():
            return named
    latest = path / "latest.pth"
    if latest.exists():
        return latest
    ckpts = sorted(
        path.glob("epoch_*.pth"),
        key=lambda p: int(p.stem.split("_")[-1]) if p.stem.split("_")[-1].isdigit() else -1,
    )
    return ckpts[-1] if ckpts else None


def keep_single_ckpt(path: Path, ckpt_name: str):
    path = Path(path)
    src = latest_ckpt(path, ckpt_name)
    if src is None or not src.exists():
        print("[WARN NO TRAIN CKPT]", path)
        return None
    final = path / ckpt_name
    if src.resolve() != final.resolve():
        tmp = final.with_suffix(final.suffix + ".tmp")
        shutil.copy2(src, tmp)
        tmp.replace(final)
    for p in path.glob("*.pth"):
        if p.name != ckpt_name:
            p.unlink()
    print("[SINGLE CKPT]", final)
    return final


def count_files(path: Path):
    return sum(1 for p in path.rglob("*") if p.is_file()) if path.exists() else 0



def cfg_list(values):
    return "[" + ",".join(str(float(value)) for value in values) + "]"


def evaluator_options():
    if not EVAL_IOU_THRS:
        return []
    iou_thrs = cfg_list(EVAL_IOU_THRS)
    return [
        f"val_evaluator.iou_thrs={iou_thrs}",
        f"test_evaluator.iou_thrs={iou_thrs}",
    ]

def cfg_options(train=True):
    return [
        f"train_dataloader.batch_size={resolved_batch_size()}",
        f"train_dataloader.num_workers={resolved_num_workers()}",
        f"val_dataloader.batch_size={resolved_batch_size()}",
        f"val_dataloader.num_workers={resolved_num_workers()}",
        f"test_dataloader.batch_size={resolved_batch_size()}",
        f"test_dataloader.num_workers={resolved_num_workers()}",
        "default_hooks.checkpoint.max_keep_ckpts=1",
    ]


def run_train_job(name, job, root):
    train_dir = root / "train"
    done = root / ".train.done"
    ckpt_name = ckpt_name_for(job)

    if not RUN_TRAIN:
        print("[SKIP TRAIN]", name)
        return latest_ckpt(train_dir, ckpt_name)

    ckpt = latest_ckpt(train_dir, ckpt_name)
    if done.exists() and ckpt and not FORCE:
        print("[SKIP TRAIN DONE]", name, ckpt)
        return keep_single_ckpt(train_dir, ckpt_name) or ckpt

    train_dir.mkdir(parents=True, exist_ok=True)
    attempt = 0
    active_load_from = globals().get("LOAD_FROM")
    resume_after_failure = False
    while True:
        use_resume = bool(ckpt) and (not FORCE or resume_after_failure)
        runtime_cfg = write_runtime_config(
            job, root, load_from=None if use_resume else active_load_from)
        cmd = ddp_cmd("tools/train.py", str(runtime_cfg), "--work-dir", str(train_dir))
        if use_resume:
            print("[RESUME TRAIN]", job["title"], ckpt)
            cmd.append("--resume")
        else:
            print("[RUN TRAIN]", job["title"])
        cmd += ["--cfg-options", *cfg_options(train=True)]

        try:
            run(cmd)
            break
        except subprocess.CalledProcessError:
            can_retry = (
                bool(AUTO_LR_RETRY_ON_FAIL)
                and isinstance(LR, str)
                and LR.lower() == "auto"
                and attempt < int(AUTO_LR_MAX_RETRIES)
            )
            if not can_retry:
                raise
            attempt += 1
            globals()["AUTO_LR_SCALE"] = float(AUTO_LR_SCALE) * float(AUTO_LR_RETRY_FACTOR)
            failed_ckpt = latest_ckpt(train_dir)
            if failed_ckpt and failed_ckpt.exists():
                ckpt = failed_ckpt
                active_load_from = None
                resume_after_failure = True
                print(f"[AUTO LR RETRY] resume from saved checkpoint: {ckpt}")
            else:
                print("[AUTO LR RETRY] no saved checkpoint found; reload initial weights")
            print(f"[AUTO LR RETRY] retry {attempt}/{AUTO_LR_MAX_RETRIES} with peak_lr={resolved_lr()}")

    final = keep_single_ckpt(train_dir, ckpt_name)
    done.touch()
    return final


def resolve_ckpt(name, job, root):
    trained = latest_ckpt(root / "train", ckpt_name_for(job))
    if trained:
        return trained

    load_from = globals().get("LOAD_FROM")
    if load_from and Path(load_from).exists():
        return Path(load_from)

    if CKPT_INPUT is None:
        return None
    external = CKPT_DIR / job["ckpt"]
    return external if external.exists() else None


def run_test_job(name, job, root, ckpt):
    test_dir = root / "test"
    pred = test_dir / "predictions.pkl"
    done = root / ".test.done"
    if not RUN_TEST:
        print("[SKIP TEST]", name)
        return pred
    if done.exists() and pred.exists() and not FORCE:
        print("[SKIP TEST DONE]", name)
        return pred
    if not ckpt or not Path(ckpt).exists():
        print("[SKIP TEST NO CKPT]", name)
        return pred
    test_dir.mkdir(parents=True, exist_ok=True)
    cmd = ddp_cmd(
        "tools/test.py", str(write_runtime_config(job, root)), str(ckpt),
        "--work-dir", str(test_dir), "--out", str(pred),
    ) + [
        "--cfg-options",
        f"test_dataloader.batch_size={resolved_batch_size()}",
        f"test_dataloader.num_workers={resolved_num_workers()}",
    ]
    print("[RUN TEST]", job["title"])
    run(cmd)
    done.touch()
    return pred


def run_optional_analysis(name, job, root, ckpt, pred):
    chart_dir = root / "charts"
    bbox_dir = root / "images" / "bboxes"
    image_source = REPO_DIR / job["image_source"]

    if RUN_CHARTS and not (root / ".charts.done").exists():
        plot_script = REPO_DIR / "tools/analysis_tools/plot_rqformer_charts.py"
        logs = list((root / "train").rglob("*.json")) + list((root / "test").rglob("*.json"))
        if plot_script.exists() and logs:
            chart_dir.mkdir(parents=True, exist_ok=True)
            run([sys.executable, str(plot_script), *map(str, logs), "--out-dir", str(chart_dir), "--title", job["title"]])
            (root / ".charts.done").touch()
        else:
            print("[SKIP CHARTS] missing script or json logs")

    if RUN_CONFUSION and not (root / ".confusion.done").exists():
        cm_script = REPO_DIR / "tools/analysis_tools/confusion_matrix.py"
        if cm_script.exists() and pred.exists():
            out_dir = chart_dir / "confusion_matrix"
            out_dir.mkdir(parents=True, exist_ok=True)
            run([
                sys.executable, str(cm_script), job["config"], str(pred), str(out_dir),
                "--score-thr", str(SCORE_THR),
                "--cfg-options",
                f"test_dataloader.batch_size={resolved_batch_size()}",
                f"test_dataloader.num_workers={resolved_num_workers()}",
            ])
            (root / ".confusion.done").touch()
        else:
            print("[SKIP CONFUSION] missing script or predictions")

    if RUN_BBOX and (FORCE or not (root / ".bbox.done").exists()):
        bbox_script = REPO_DIR / "tools/analysis_tools/visualize_rqformer_bboxes.py"
        if bbox_script.exists() and ckpt and Path(ckpt).exists() and image_source.exists():
            bbox_dir.mkdir(parents=True, exist_ok=True)
            bbox_cmd = [
                sys.executable, str(bbox_script), job["config"], str(ckpt), str(image_source),
                "--out-dir", str(bbox_dir), "--device", "cuda:0", "--score-thr", str(SCORE_THR),
                "--max-images", str(SAMPLE_IMAGES),
            ]
            if BBOX_GROUP_BY_CLASS:
                bbox_cmd += [
                    "--group-by-class",
                    "--images-per-class", str(BBOX_IMAGES_PER_CLASS),
                    "--candidates-per-class", str(BBOX_CANDIDATES_PER_CLASS),
                ]
            run(bbox_cmd)
            (root / ".bbox.done").touch()
        else:
            print("[SKIP BBOX] missing script, ckpt, or image source")


    counts = {
        "charts": count_files(chart_dir),
        "bboxes": count_files(bbox_dir),
    }
    print("Charts:", counts["charts"], "BBoxes:", counts["bboxes"])
    return counts


## 6.1 Load Checkpoint

Dùng `RESUME_CKPT_PATH` để chỉ định checkpoint upload lên Kaggle. Mặc định load weight từ checkpoint nhưng reset optimizer/scheduler để ổn định hơn.


In [ ]:
RESUME_CKPT_PATH = Path("/kaggle/input/datasets/aichcmc2024006/data-epoch/epoch_2.pth")
RESUME_INPUT_EPOCH = 2
USE_RESUME_CKPT = True
RESET_OPTIMIZER_WHEN_LOADING = True
LOAD_FROM = None

checkpoint_report = {"use_checkpoint": USE_RESUME_CKPT, "source": str(RESUME_CKPT_PATH)}

if USE_RESUME_CKPT:
    if not RESUME_CKPT_PATH.exists():
        raise FileNotFoundError(RESUME_CKPT_PATH)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    local_ckpt = CKPT_DIR / RESUME_CKPT_PATH.name
    shutil.copy2(RESUME_CKPT_PATH, local_ckpt)

    if RESET_OPTIMIZER_WHEN_LOADING:
        LOAD_FROM = str(local_ckpt)
        FORCE = 1
        checkpoint_report.update({
            "mode": "load_from_weights_only",
            "local_checkpoint": str(local_ckpt),
            "effective_start_epoch": RESUME_INPUT_EPOCH,
            "log_epoch_note": "training log restarts from epoch 1",
        })
    else:
        train_dir = WORK_ROOT / "rroiformer" / "dior" / "train"
        train_dir.mkdir(parents=True, exist_ok=True)
        resume_target = train_dir / RESUME_CKPT_PATH.name
        shutil.copy2(RESUME_CKPT_PATH, resume_target)
        (train_dir / "last_checkpoint").write_text(str(resume_target), encoding="utf-8")
        FORCE = 0
        checkpoint_report.update({"mode": "resume_optimizer_scheduler", "local_checkpoint": str(resume_target)})
else:
    checkpoint_report.update({"mode": "train_from_scratch"})

pd.DataFrame([checkpoint_report])


## 7. Chạy Workflow

Cell chính để chạy train, test, confusion matrix, bbox visualization, bbox và tổng hợp số lượng output.


In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = resolved_device()
WORK_ROOT.mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = None

import math
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

_WORKFLOW_IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def workflow_list_files(path: Path, exts=None):
    if not path.exists():
        return []
    files = [item for item in path.rglob("*") if item.is_file()]
    if exts is not None:
        files = [item for item in files if item.suffix.lower() in exts]
    return sorted(files)


def workflow_bbox_files(root: Path):
    by_class = root / "images" / "bboxes" / "by_class"
    if by_class.exists():
        files = []
        for class_dir in sorted([item for item in by_class.iterdir() if item.is_dir()]):
            files.extend(workflow_list_files(class_dir, _WORKFLOW_IMAGE_EXTS)[:BBOX_IMAGES_PER_CLASS])
        return files
    return workflow_list_files(root / "images" / "bboxes", _WORKFLOW_IMAGE_EXTS)[:SAMPLE_IMAGES]


def workflow_grid_title(path: Path):
    parts = path.parts
    if "by_class" in parts:
        idx = parts.index("by_class")
        if idx + 1 < len(parts):
            return f"{parts[idx + 1]} | {path.stem}"
    return path.stem

def workflow_confusion_files(files):
    return [path for path in files if "confusion" in str(path).lower()]


def workflow_show_large_centered(title, files):
    if not files:
        return
    for path in files:
        fig, ax = plt.subplots(figsize=(13.5, 12.5), dpi=180)
        try:
            ax.imshow(mpimg.imread(path))
            ax.set_title(title, fontsize=20, fontweight="bold", pad=18, loc="center")
        except Exception as exc:
            ax.text(0.5, 0.5, f"Cannot load\n{path.name}\n{exc}", ha="center", va="center", fontsize=12)
        ax.axis("off")
        fig.tight_layout(pad=1.0)
        plt.show()



def workflow_show_grid(title, files, cols=4, max_per_figure=16):
    if not files:
        print(f"[NO IMAGE] {title}")
        return
    cols = max(1, min(cols, max_per_figure))
    for batch in [files[i:i + max_per_figure] for i in range(0, len(files), max_per_figure)]:
        rows = math.ceil(len(batch) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 6.2, rows * 5.0), dpi=160)
        axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
        fig.suptitle(title, fontsize=18, fontweight="bold")
        for ax, path in zip(axes, batch):
            try:
                ax.imshow(mpimg.imread(path))
                ax.set_title(workflow_grid_title(path), fontsize=12)
            except Exception as exc:
                ax.text(0.5, 0.5, f"Cannot load\n{path.name}\n{exc}", ha="center", va="center", fontsize=8)
            ax.axis("off")
        for ax in axes[len(batch):]:
            ax.axis("off")
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()


def workflow_show_outputs(name):
    root = WORK_ROOT / "rroiformer" / name
    chart_files = workflow_list_files(root / "charts", _WORKFLOW_IMAGE_EXTS)
    bbox_files = workflow_bbox_files(root)

    rows = []
    for group, files in {
        "charts": chart_files,
        "bbox_images": bbox_files,
    }.items():
        rows.append({
            "job": name,
            "group": group,
            "count": len(files),
            "latest": str(max(files, key=lambda p: p.stat().st_mtime)) if files else "-",
        })
        for path in files[:50]:
            print(f"[{group}] {path}")

    if display:
        display(pd.DataFrame(rows))
    else:
        print(pd.DataFrame(rows).to_string(index=False))

    confusion_files = workflow_confusion_files(chart_files)
    other_chart_files = [path for path in chart_files if path not in confusion_files]
    workflow_show_large_centered(f"{JOBS[name]['dataset_label']} | Confusion Matrix", confusion_files)
    workflow_show_grid(f"{JOBS[name]['dataset_label']} | Training Charts", other_chart_files, cols=2, max_per_figure=4)
    workflow_show_grid(
        f"{JOBS[name]['dataset_label']} | Bounding Box Visualization",
        bbox_files,
        cols=BBOX_GRID_COLS,
        max_per_figure=BBOX_IMAGES_PER_FIGURE,
    )


run_rows = []
if display:
    display(runtime_table())
else:
    print(runtime_table().to_string(index=False))

for name in selected_names():
    job = JOBS[name]
    root = WORK_ROOT / "rroiformer" / name
    root.mkdir(parents=True, exist_ok=True)

    cfg = REPO_DIR / job["config"]
    data = REPO_DIR / job["data_link"]
    if not cfg.exists():
        raise FileNotFoundError(cfg)
    if not data.exists():
        raise FileNotFoundError(data)

    trained = run_train_job(name, job, root)
    ckpt = resolve_ckpt(name, job, root)
    pred = run_test_job(name, job, root, ckpt)
    analysis = run_optional_analysis(name, job, root, ckpt, pred) or {"charts": 0, "bboxes": 0}

    run_rows.append({
        "job": name,
        "checkpoint": str(ckpt) if ckpt else "-",
        "predictions": str(pred) if pred.exists() else "-",
        **analysis,
    })

run_table = pd.DataFrame(run_rows)
if display:
    display(run_table)
else:
    print(run_table.to_string(index=False))

if AUTO_DISPLAY_OUTPUTS:
    for name in selected_names():
        if Markdown and display:
            display(Markdown(f"### Output hiển thị tự động: {JOBS[name]['dataset_label']}"))
        else:
            print(f"Output hiển thị tự động: {JOBS[name]['dataset_label']}")
        workflow_show_outputs(name)


## 8. Bảng Summary

Tạo bảng kết quả theo format báo cáo: Dataset, AP50, AP75, mAP, Backbone, lr schd, batch, Angle, Query và Configs.


In [ ]:
from IPython.display import Markdown, display

summary_script = REPO_DIR / "tools/analysis_tools/write_rqformer_summary.py"
summary_path = WORK_ROOT / "rroiformer" / "summary_results.md"

if RUN_SUMMARY and summary_script.exists():
    run([
        sys.executable, str(summary_script),
        "--repo-dir", str(REPO_DIR),
        "--work-root", str(WORK_ROOT),
        "--out", str(summary_path),
    ])
    display(Markdown(summary_path.read_text(encoding="utf-8", errors="replace")))
else:
    pd.DataFrame([{"summary": "skipped", "reason": "script missing or RUN_SUMMARY=0"}])


## 8.1 Biểu Đồ Báo Cáo

Đọc JSON log và vẽ dashboard gồm Loss, Learning Rate, Accuracy và AP50/AP75/mAP. Không tự bịa validation loss nếu log không có.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt


REPORT_DPI = 220


def read_mmengine_json_logs(log_root: Path):
    records = []
    for path in sorted(log_root.rglob("*.json")):
        try:
            lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        except Exception:
            continue
        for line in lines:
            line = line.strip()
            if not line.startswith("{"):
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            row["_log_file"] = str(path)
            records.append(row)
    return records


def pick_metric(row, names):
    for name in names:
        value = row.get(name)
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            return float(value)
    return None


def row_epoch(row, fallback):
    epoch = row.get("epoch")
    if isinstance(epoch, (int, float)):
        return float(epoch)
    return float(fallback)


def row_x(row, fallback):
    epoch = row.get("epoch")
    step = row.get("iter", row.get("step"))
    if isinstance(epoch, (int, float)):
        if isinstance(step, (int, float)):
            return float(epoch) + float(step) / 1_000_000.0
        return float(epoch)
    if isinstance(step, (int, float)):
        return float(step)
    return float(fallback)


def build_curve_tables(name):
    root = WORK_ROOT / "rroiformer" / name
    records = read_mmengine_json_logs(root)
    train_rows, val_rows = [], []

    for idx, row in enumerate(records):
        x = row_x(row, idx)
        epoch = row_epoch(row, idx)
        loss = pick_metric(row, ["loss", "train/loss"])
        val_loss = pick_metric(row, ["val/loss", "validation/loss", "val_loss"])
        lr = pick_metric(row, ["lr", "learning_rate"])
        pos_acc = pick_metric(row, ["s1.pos_acc", "s0.pos_acc", "pos_acc", "acc"])
        ap50 = pick_metric(row, ["dota/AP50", "AP50"])
        ap75 = pick_metric(row, ["dota/AP75", "AP75"])
        mean_ap = pick_metric(row, ["dota/mAP", "mAP"])

        if loss is not None and mean_ap is None and ap50 is None and ap75 is None:
            train_rows.append({"x": x, "epoch": epoch, "loss": loss, "lr": lr, "pos_acc": pos_acc})
        if val_loss is not None or mean_ap is not None or ap50 is not None or ap75 is not None:
            val_rows.append({"x": x, "epoch": epoch, "val_loss": val_loss, "mAP": mean_ap, "AP50": ap50, "AP75": ap75})

    train_df = pd.DataFrame(train_rows)
    val_df = pd.DataFrame(val_rows)
    if not train_df.empty:
        train_df["epoch_round"] = train_df["epoch"].round().astype(int)
    if not val_df.empty:
        val_df["epoch_round"] = val_df["epoch"].round().astype(int)
    return train_df, val_df


def epoch_curve(df, value_col):
    if df.empty or value_col not in df or not df[value_col].notna().any():
        return pd.DataFrame(columns=["epoch", value_col])
    return (
        df.dropna(subset=[value_col])
        .groupby("epoch_round", as_index=False)[value_col]
        .mean()
        .rename(columns={"epoch_round": "epoch"})
    )


def plot_loss_report(name, train_df, val_df, out_dir):
    train_loss = epoch_curve(train_df, "loss")
    val_loss = epoch_curve(val_df, "val_loss")

    fig, ax = plt.subplots(figsize=(11, 7), dpi=REPORT_DPI)
    if not train_loss.empty:
        ax.plot(train_loss["epoch"], train_loss["loss"], label="train", color="#1f77b4", linewidth=2.4)
    if not val_loss.empty:
        ax.plot(val_loss["epoch"], val_loss["val_loss"], label="validation", color="#ff7f0e", linewidth=2.4)
    else:
        ax.text(
            0.5, 0.9,
            "Validation loss không có trong log; pipeline chỉ đánh giá AP/mAP ở validation.",
            transform=ax.transAxes,
            ha="center",
            fontsize=11,
            color="dimgray",
        )

    ax.set_title(f"{JOBS[name]['dataset_label']} | Train and Validation Loss", fontsize=18, fontweight="bold")
    ax.set_xlabel("Epoch", fontsize=14)
    ax.set_ylabel("Loss", fontsize=14)
    ax.tick_params(axis="both", labelsize=12)
    ax.legend(fontsize=12)
    ax.grid(alpha=0.25)
    fig.tight_layout()

    out_path = out_dir / "loss_train_validation_report.png"
    fig.savefig(out_path, dpi=REPORT_DPI, bbox_inches="tight")
    plt.show()
    return out_path


def plot_report_dashboard(name):
    train_df, val_df = build_curve_tables(name)
    out_dir = WORK_ROOT / "rroiformer" / name / "charts" / "notebook_report"
    out_dir.mkdir(parents=True, exist_ok=True)

    loss_chart = plot_loss_report(name, train_df, val_df, out_dir)

    train_loss = epoch_curve(train_df, "loss")
    lr_curve = epoch_curve(train_df.dropna(subset=["lr"]) if "lr" in train_df else pd.DataFrame(), "lr")
    acc_curve = epoch_curve(train_df.dropna(subset=["pos_acc"]) if "pos_acc" in train_df else pd.DataFrame(), "pos_acc")

    fig, axes = plt.subplots(2, 2, figsize=(18, 12), dpi=REPORT_DPI)
    fig.suptitle(f"{JOBS[name]['dataset_label']} | Training and Evaluation Dashboard", fontsize=20, fontweight="bold")

    ax = axes[0, 0]
    if not train_loss.empty:
        ax.plot(train_loss["epoch"], train_loss["loss"], color="#1f77b4", linewidth=2.2, label="train")
        ax.legend(fontsize=11)
    ax.set_title("Loss by Epoch", fontsize=15, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.25)

    ax = axes[0, 1]
    if not lr_curve.empty:
        ax.plot(lr_curve["epoch"], lr_curve["lr"], color="#ff7f0e", linewidth=2.2, label="learning rate")
        ax.legend(fontsize=11)
    ax.set_title("Learning Rate Schedule", fontsize=15, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("LR")
    ax.grid(alpha=0.25)

    ax = axes[1, 0]
    if not acc_curve.empty:
        ax.plot(acc_curve["epoch"], acc_curve["pos_acc"], color="#2ca02c", linewidth=2.2, label="pos_acc")
        ax.legend(fontsize=11)
    ax.set_title("Positive Sample Accuracy", fontsize=15, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)")
    ax.grid(alpha=0.25)

    ax = axes[1, 1]
    if not val_df.empty:
        for metric, color in [("AP50", "#9467bd"), ("AP75", "#d62728"), ("mAP", "#8c564b")]:
            metric_df = epoch_curve(val_df, metric)
            if not metric_df.empty:
                ax.plot(metric_df["epoch"], metric_df[metric], marker="o", linewidth=2.2, label=metric, color=color)
        ax.legend(fontsize=11)
    ax.set_title("Validation Metrics", fontsize=15, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Score")
    ax.grid(alpha=0.25)

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    dashboard_path = out_dir / "training_dashboard_2x2.png"
    fig.savefig(dashboard_path, dpi=REPORT_DPI, bbox_inches="tight")
    plt.show()

    summary = {
        "job": name,
        "loss_chart": str(loss_chart),
        "dashboard": str(dashboard_path),
        "train_points": len(train_df),
        "validation_points": len(val_df),
    }
    if not val_df.empty:
        last_val = val_df.dropna(how="all", subset=["mAP", "AP50", "AP75"]).tail(1)
        if not last_val.empty:
            summary.update({metric: last_val.iloc[0].get(metric) for metric in ["AP50", "AP75", "mAP"]})
    return pd.DataFrame([summary]), train_df, val_df


chart_reports = []
for _name in selected_names():
    report_df, train_curve_df, val_curve_df = plot_report_dashboard(_name)
    chart_reports.append(report_df)
    display(report_df)
    if not val_curve_df.empty:
        display(val_curve_df.tail(10))

if chart_reports:
    display(pd.concat(chart_reports, ignore_index=True))


## 9. Gallery Kết Quả

Hiển thị output bằng bảng gắn với ảnh theo dạng **grid**. Ưu tiên hiển thị **Bounding Box** theo **20 lớp của DIOR-R**, mỗi lớp gồm **2 ảnh**. Bố cục mỗi trang là **4 cột × 2 hàng**, trình bày rõ ràng, đồng đều và dễ quan sát.


In [ ]:
from pathlib import Path
import math

try:
    from IPython.display import Markdown, display
except Exception:
    display = None
    Markdown = None

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
TEXT_EXTS = {".log", ".txt", ".md", ".json"}


def list_files(path: Path, exts=None):
    if not path.exists():
        return []
    files = [item for item in path.rglob("*") if item.is_file()]
    if exts is not None:
        files = [item for item in files if item.suffix.lower() in exts]
    return sorted(files)




def bbox_gallery_files(root: Path):
    by_class = root / "images" / "bboxes" / "by_class"
    if by_class.exists():
        files = []
        for class_dir in sorted([item for item in by_class.iterdir() if item.is_dir()]):
            files.extend(list_files(class_dir, IMAGE_EXTS)[:BBOX_IMAGES_PER_CLASS])
        return files
    return list_files(root / "images" / "bboxes", IMAGE_EXTS)[:SAMPLE_IMAGES]


def image_grid_title(path: Path):
    parts = path.parts
    if "by_class" in parts:
        idx = parts.index("by_class")
        if idx + 1 < len(parts):
            return f"{parts[idx + 1]} | {path.stem}"
    return path.stem

def confusion_matrix_files(files):
    return [path for path in files if "confusion" in str(path).lower()]


def display_large_centered_image(title, files):
    if not files:
        return
    for path in files:
        fig, ax = plt.subplots(figsize=(13.5, 12.5), dpi=180)
        try:
            ax.imshow(mpimg.imread(path))
            ax.set_title(title, fontsize=20, fontweight="bold", pad=18, loc="center")
        except Exception as exc:
            ax.text(0.5, 0.5, f"Cannot load\n{path.name}\n{exc}", ha="center", va="center", fontsize=12)
        ax.axis("off")
        fig.tight_layout(pad=1.0)
        plt.show()


def output_inventory(name):
    root = WORK_ROOT / "rroiformer" / name
    groups = {
        "checkpoints": list_files(root / "train", {".pth"}),
        "predictions": list_files(root / "test", {".pkl"}),
        "charts": list_files(root / "charts", IMAGE_EXTS),
        "bbox_images": list_files(root / "images" / "bboxes", IMAGE_EXTS),
        "logs": list_files(root, TEXT_EXTS),
    }
    rows = []
    for group, files in groups.items():
        rows.append({
            "job": name,
            "group": group,
            "count": len(files),
            "latest": str(max(files, key=lambda p: p.stat().st_mtime)) if files else "-",
        })
    return pd.DataFrame(rows), groups


def file_table(files, group, limit=20):
    rows = []
    for path in files[:limit]:
        rows.append({
            "group": group,
            "file": path.name,
            "size_mb": round(path.stat().st_size / 1024**2, 3),
            "path": str(path),
        })
    return pd.DataFrame(rows)


def chunked(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]


def display_image_grid(title, files, cols=4, max_per_figure=16):
    if not files:
        return
    cols = max(1, min(cols, max_per_figure))
    for batch in chunked(files, max_per_figure):
        rows = math.ceil(len(batch) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 6.2, rows * 5.0), dpi=160)
        axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
        fig.suptitle(title, fontsize=18, fontweight="bold")

        for ax, path in zip(axes, batch):
            try:
                image = mpimg.imread(path)
                ax.imshow(image)
                ax.set_title(image_grid_title(path), fontsize=12)
            except Exception as exc:
                ax.text(0.5, 0.5, f"Cannot load\n{path.name}\n{exc}", ha="center", va="center", fontsize=8)
            ax.axis("off")

        for ax in axes[len(batch):]:
            ax.axis("off")

        fig.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()


def show_report_outputs(name):
    root = WORK_ROOT / "rroiformer" / name
    inventory, groups = output_inventory(name)
    if display and Markdown:
        display(Markdown(f"### {JOBS[name]['dataset_label']} Outputs"))
    display(inventory)

    important_files = []
    for group in ["checkpoints", "predictions", "charts"]:
        important_files.append(file_table(groups[group], group, limit=20))
    important_files = pd.concat(important_files, ignore_index=True) if important_files else pd.DataFrame()
    if not important_files.empty:
        display(important_files)

    confusion_files = confusion_matrix_files(groups["charts"])
    other_chart_files = [path for path in groups["charts"] if path not in confusion_files]
    display_large_centered_image(f"{JOBS[name]['dataset_label']} | Confusion Matrix", confusion_files)
    display_image_grid(
        f"{JOBS[name]['dataset_label']} | Training Charts",
        other_chart_files,
        cols=2,
        max_per_figure=4,
    )

    bbox_files = bbox_gallery_files(root)
    bbox_meta_rows = []
    for path in bbox_files:
        parts = path.parts
        cls = parts[parts.index("by_class") + 1] if "by_class" in parts else "-"
        bbox_meta_rows.append({"class": cls, "image": path.stem.replace("_bbox", ""), "file": path.name})
    if bbox_meta_rows:
        display(pd.DataFrame(bbox_meta_rows))

    display_image_grid(
        f"{JOBS[name]['dataset_label']} | Bounding Box Visualization",
        bbox_gallery_files(root),
        cols=BBOX_GRID_COLS,
        max_per_figure=BBOX_IMAGES_PER_FIGURE,
    )


for _name in selected_names():
    show_report_outputs(_name)

summary_path = WORK_ROOT / "rroiformer" / "summary_results.md"
if summary_path.exists() and display and Markdown:
    display(Markdown("### Summary Results"))
    display(Markdown(summary_path.read_text(encoding="utf-8", errors="replace")))


## 10. Zip Và Download

Zip toàn bộ output thành một file duy nhất và tạo link tải trực tiếp trong Kaggle notebook.


In [ ]:
from IPython.display import FileLink


def tree_rows(path: Path, max_depth=4):
    if not path.exists():
        return []
    base_depth = len(path.parts)
    rows = []
    for item in sorted(path.rglob("*")):
        depth = len(item.parts) - base_depth
        if depth > max_depth:
            continue
        rows.append({
            "depth": depth,
            "type": "dir" if item.is_dir() else "file",
            "path": str(item.relative_to(path)),
            "size_mb": round(item.stat().st_size / 1024**2, 3) if item.is_file() else None,
        })
    return rows

output_root = WORK_ROOT / "rroiformer"
display(pd.DataFrame(tree_rows(output_root, max_depth=4)))

archive_base = Path("/kaggle/working/rqformer2_full_workflow_outputs")
zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(WORK_ROOT))
zip_file = Path(zip_path)

display(pd.DataFrame([{
    "archive": str(zip_file),
    "size_mb": round(zip_file.stat().st_size / 1024**2, 2),
}]))
display(FileLink(str(zip_file)))


## Gợi Ý Chạy Nhanh

Một vài cấu hình hay dùng: đổi checkpoint, train từ đầu hoặc chỉ test/visualize lại từ checkpoint hiện có.

```python
RESUME_CKPT_PATH = Path("/kaggle/input/<dataset>/epoch_10.pth")
RESUME_INPUT_EPOCH = 10
USE_RESUME_CKPT = True
RESET_OPTIMIZER_WHEN_LOADING = True
```

```python
USE_RESUME_CKPT = False
FORCE = 1
```

```python
RUN_SETUP = 0
RUN_TRAIN = 0
RUN_TEST = 1
RUN_CHARTS = 1
RUN_CONFUSION = 1
RUN_BBOX = 1
RUN_SUMMARY = 1
```
